In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.signal import butter, lfilter
from scipy.signal import freqz

import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
subjectData = {}
subjectDataEVAL = {}

In [ ]:
ns = 10
for i in range(1,ns):
    #data_path = os.path.join("D://", '/Research/EEG/BCI_Competition_IV_2a/Dataset/NPZ/A{:02d}T.npz'.format(i))
    data_path = os.path.join('/content/drive/MyDrive/BCI','A{:02d}T.npz'.format(i))
    subject = 'subject{:02d}'.format(i)

    subjectData[subject] = np.load(data_path)

In [ ]:
print(len(subjectData))

9


In [ ]:
ns = 10
for i in range(1,ns):
    #data_path = os.path.join("D://", '/Research/EEG/BCI_Competition_IV_2a/Dataset/NPZ/A{:02d}T.npz'.format(i))
    data_path = os.path.join('/content/drive/MyDrive/BCI','A{:02d}E.npz'.format(i))
    subject = 'subject{:02d}'.format(i)

    subjectDataEVAL[subject] = np.load(data_path)

In [ ]:
print(type(subjectData['subject01']['etyp']))

<class 'numpy.ndarray'>


In [ ]:
print(type(subjectData['subject01']))
print(subjectData['subject01'].files)

<class 'numpy.lib.npyio.NpzFile'>
['s', 'etyp', 'epos', 'edur', 'artifacts']


s: contains time-series recorded EEG signals of shape MxN array. N is the number of electrodes (22 EEG and 3 EOG), M can vary

etpye: event type to indicate event occurence

epos: event position denoting event start at s


edur: event duration artifacts: size of 288x1, 6x48 = 288 where 6 is the number of runs with 48 trials. 48 trials have 12 trials of 4 class each

In [ ]:
print('Sample\t Electrodes')
for i in range(1,ns):
    sub = 'subject{:02d}'.format(i)
    print(subjectData[sub]['s'].shape)

Sample	 Electrodes
(672528, 25)
(677169, 25)
(660530, 25)
(600915, 25)
(686120, 25)
(678980, 25)
(681071, 25)
(675270, 25)
(673328, 25)


In [ ]:
# Bandpass filter
def butter_bandpass_filter(signal, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut/nyq
    high = highcut/nyq
    b, a = butter(order, [low, high], btype='band')
    y = lfilter(b,a,signal, axis=-1)
    return y

In [ ]:
def butterFilter(cleanData, data, sub):
    lowcut = 4
    highcut = 40
    fs = 250
    print('Processing', sub)
    cleanData[sub] = butter_bandpass_filter(data, lowcut, highcut, fs)

In [ ]:
# Transpose the raw EEG data to apply filtering on the time-series axis
rawData = {}
for i in range(1,ns):
    sub = 'subject{:02d}'.format(i)
    rawData[sub] = subjectData[sub]['s'].T

    print(rawData[sub].shape)

(25, 672528)
(25, 677169)
(25, 660530)
(25, 600915)
(25, 686120)
(25, 678980)
(25, 681071)
(25, 675270)
(25, 673328)


In [ ]:
cleanEEG = {}
for i in range(1,ns):
    sub = 'subject{:02d}'.format(i)
    butterFilter(cleanEEG, rawData[sub], sub)

    print(cleanEEG[sub].shape)

Processing subject01
(25, 672528)
Processing subject02
(25, 677169)
Processing subject03
(25, 660530)
Processing subject04
(25, 600915)
Processing subject05
(25, 686120)
Processing subject06
(25, 678980)
Processing subject07
(25, 681071)
Processing subject08
(25, 675270)
Processing subject09
(25, 673328)


There are 4 classes with event code as 769, 770, 771, 772 for left, right, foot and tongue.

In [ ]:
left_class_code = 769
right_class_code = 770
foot_class_code = 771
tongue_class_code = 772

for i in range(1,2):
    sub = 'subject{:02d}'.format(i)
    left = subjectData[sub]['epos'][subjectData[sub]['etyp'] == left_class_code]
    right = subjectData[sub]['epos'][subjectData[sub]['etyp'] == right_class_code]
    foot = subjectData[sub]['epos'][subjectData[sub]['etyp'] == foot_class_code]
    tongue = subjectData[sub]['epos'][subjectData[sub]['etyp'] == tongue_class_code]



In [ ]:
left

array([ 98242, 100250, 112163, 114058, 116030, 124069, 126135, 130237,
       140283, 154531, 168776, 186876, 189204, 191207, 197085, 201113,
       208998, 214809, 229142, 239079, 245162, 251366, 261676, 283711,
       289960, 297948, 299998, 313611, 321841, 333953, 346094, 348201,
       354427, 364371, 376373, 380546, 384877, 386795, 398764, 402668,
       404563, 430788, 442929, 447128, 451262, 455346, 459281, 469374,
       481712, 489701, 491618, 505314, 529584, 537693, 541871, 546054,
       548097, 550161, 552181, 572157, 576544, 600205, 622505, 626419,
       632502, 636599, 642889, 646996, 654876, 663044, 666878, 671051],
      dtype=int32)

72

These are the sample points where the event has started

Steps to start with

1. Remove EOG channels from all subjects

2. Crop the data for each subject such that you get the data during the motor imagery peroid (refer Experimental_Paradigm)


In [ ]:
length = len(right) + len(left) + len(foot) + len(tongue)
print(length)

288


# TRYING TO REPLICATE THE JESUS GITHUB STUFF

In [ ]:
d = np.load('/content/drive/MyDrive/BCI/A01E.npz')
print(np.unique(d['etyp']))


[  276   277   768   783  1023  1072 32766]


In [ ]:
from FBCSP_Multiclass import FBCSP_Multiclass

In [ ]:
from sklearn.model_selection import train_test_split

def make_trials_dict(subject):
    s    = subject['s'][:, :22].astype(np.float64)
    epos = subject['epos'].flatten().astype(int)
    etyp = subject['etyp'].flatten().astype(int)
    win  = np.arange(0, 4 * 250)

    buckets = {'left': [], 'right': [], 'feet': [], 'tongue': []}
    code_to_name = {769: 'left', 770: 'right', 771: 'feet', 772: 'tongue'}

    for pos, code in zip(epos, etyp):
        if code in code_to_name:
            buckets[code_to_name[code]].append(s[pos + win, :].T)

    return {k: np.stack(v) for k, v in buckets.items()}

"""accuracies = []
for idx in range(1, 10):
    sub = f'subject{idx:02d}'
    full_dict = make_trials_dict(subjectData[sub])

    train_dict, test_dict = {}, {}
    y_test_all = []
    for lbl, (cls, arr) in enumerate(full_dict.items(), 1):
        n = arr.shape[0]
        split = int(n * 0.8)
        train_dict[cls] = arr[:split]
        test_dict[cls]  = arr[split:]
        y_test_all.extend([lbl] * (n - split))

    clf = FBCSP_Multiclass(train_dict, 250, print_var=False)

    X_test = np.concatenate([test_dict[cls] for cls in ['left','right','feet','tongue']])
    y_test  = np.array(y_test_all)
    y_pred  = clf.evaluateTrial(X_test)

    acc = np.mean(y_pred == y_test)
    accuracies.append(acc)
    print(f'Subject {idx:02d}: {acc:.4f}')

print(f'\nMean: {np.mean(accuracies):.4f}')"""

from scipy.io import loadmat

accuracies = []
for idx in range(1, 10):
    sub = f'subject{idx:02d}'

    train_dict = make_trials_dict(subjectData[sub])

    # Extract eval trials using event 783
    s    = subjectDataEVAL[sub]['s'][:, :22].astype(np.float64)
    epos = subjectDataEVAL[sub]['epos'].flatten().astype(int)
    etyp = subjectDataEVAL[sub]['etyp'].flatten().astype(int)
    win  = np.arange(0, 4 * 250)

    cue_pos = epos[etyp == 783]
    X_test  = np.stack([s[pos + win, :].T for pos in cue_pos])

    # Load true labels
    #mat = loadmat(f'/content/drive/MyDrive/BCI/A{idx:02d}E.mat')
    mat = loadmat(f'/content/A{idx:02d}E.mat')
    y_test = mat['classlabel'].flatten()   # values 1-4

    clf    = FBCSP_Multiclass(train_dict, 250, print_var=False)
    y_pred = clf.evaluateTrial(X_test)

    acc = np.mean(y_pred == y_test)
    accuracies.append(acc)
    print(f'Subject {idx:02d}: {acc:.4f}')

print(f'\nMean: {np.mean(accuracies):.4f}')


Subject 01: 0.7014
Subject 02: 0.5417
Subject 03: 0.7951
Subject 04: 0.6007
Subject 05: 0.5104
Subject 06: 0.3785
Subject 07: 0.8021
Subject 08: 0.7118
Subject 09: 0.6840

Mean: 0.6362


## FBCSP Baseline — Implementation Settings

**Implementation:** Zancanaro (jesus-333/FBCSP-Python), FBCSP_V4 + FBCSP_Multiclass

### Data
| Setting | Value |
|---|---|
| Dataset | BCI Competition IV Dataset 2a |
| Channels | 22 EEG (EOG dropped) |
| Sampling rate | 250 Hz |
| Epoch window | 0–4 s from cue onset (events 769–772) = 1000 samples |
| Train | A0xT.npz (288 trials/subject) |
| Test | A0xE.npz + true labels from A0xE.mat (288 trials/subject) |
| Artifacts | Not removed (all 288 trials kept) |

### Filter Bank
| Setting | Value |
|---|---|
| Filter type | Butterworth bandpass, zero-phase (filtfilt) |
| Filter order | 3 |
| Frequency bands | np.linspace(4, 40, 10) → 9 bands: [4–8], [8–12], [12–16], [16–20], [20–24], [24–28], [28–32], [32–36], [36–40] Hz |

### CSP
| Setting | Value |
|---|---|
| Components (n_w) | 2 → selects first 2 + last 2 = 4 filters per band |
| Feature | log-variance of 4 selected components |
| Feature selection | MIBIF, top n_features=4 per band |

### Classification
| Setting | Value |
|---|---|
| Classifier | LDA (default) |
| Multi-class strategy | One-vs-Rest (OVR): 4 binary classifiers |

### Results
| Subject | Accuracy |
|---|---|
| 01 | 0.6771 |
| 02 | 0.5417 |
| 03 | 0.8056 |
| 04 | 0.5868 |
| 05 | 0.5069 |
| 06 | 0.4236 |
| 07 | 0.7708 |
| 08 | 0.7188 |
| 09 | 0.6771 |
| **Mean** | **0.6343** |


# NOW TRYING IT FOR CNN

In [ ]:
#import sys
#sys.path.insert(0, '/content/drive/MyDrive/BCI/EEGNet')
from EEGNet import EEGNetModel

#model = EEGNetModel(chans=22, classes=4, time_points=1000)
model = EEGNetModel(chans=22, classes=4, time_points=512, temp_kernel=64)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # should print 'cuda' if GPU runtime is enabled

cuda


In [ ]:
"""from scipy.signal import resample_poly

def subject_to_tensors(npz_train, npz_eval, mat_path):
    train_dict = make_trials_dict(npz_train)

    X_train = np.concatenate([train_dict[cls] for cls in ['left','right','feet','tongue']])
    y_train = np.concatenate([
        np.full(train_dict[cls].shape[0], lbl)
        for cls, lbl in zip(['left','right','feet','tongue'], [0,1,2,3])
    ])

    s    = npz_eval['s'][:, :22].astype(np.float64)
    epos = npz_eval['epos'].flatten().astype(int)
    etyp = npz_eval['etyp'].flatten().astype(int)
    win  = np.arange(0, 4 * 250)
    X_eval = np.stack([s[pos + win, :].T for pos in epos[etyp == 783]])
    y_eval = loadmat(mat_path)['classlabel'].flatten() - 1

    # Resample 250 Hz -> 128 Hz (4s: 1000 samples -> 512 samples)
    X_train = resample_poly(X_train, up=64, down=125, axis=2)
    X_eval  = resample_poly(X_eval,  up=64, down=125, axis=2)

    X_train = torch.tensor(X_train[:, np.newaxis], dtype=torch.float32)
    X_eval  = torch.tensor(X_eval[:, np.newaxis],  dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_eval  = torch.tensor(y_eval,  dtype=torch.long)

    return X_train, y_train, X_eval, y_eval"""


"from scipy.signal import resample_poly\n\ndef subject_to_tensors(npz_train, npz_eval, mat_path):\n    train_dict = make_trials_dict(npz_train)\n\n    X_train = np.concatenate([train_dict[cls] for cls in ['left','right','feet','tongue']])\n    y_train = np.concatenate([\n        np.full(train_dict[cls].shape[0], lbl)\n        for cls, lbl in zip(['left','right','feet','tongue'], [0,1,2,3])\n    ])\n\n    s    = npz_eval['s'][:, :22].astype(np.float64)\n    epos = npz_eval['epos'].flatten().astype(int)\n    etyp = npz_eval['etyp'].flatten().astype(int)\n    win  = np.arange(0, 4 * 250)\n    X_eval = np.stack([s[pos + win, :].T for pos in epos[etyp == 783]])\n    y_eval = loadmat(mat_path)['classlabel'].flatten() - 1\n\n    # Resample 250 Hz -> 128 Hz (4s: 1000 samples -> 512 samples)\n    X_train = resample_poly(X_train, up=64, down=125, axis=2)\n    X_eval  = resample_poly(X_eval,  up=64, down=125, axis=2)\n\n    X_train = torch.tensor(X_train[:, np.newaxis], dtype=torch.float32)\n    

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.signal import resample_poly

def subject_to_tensors(npz_train, npz_eval, mat_path):
    train_dict = make_trials_dict(npz_train)

    X_train = np.concatenate([train_dict[cls] for cls in ['left','right','feet','tongue']])
    y_train = np.concatenate([
        np.full(train_dict[cls].shape[0], lbl)
        for cls, lbl in zip(['left','right','feet','tongue'], [0,1,2,3])
    ])

    s    = npz_eval['s'][:, :22].astype(np.float64)
    epos = npz_eval['epos'].flatten().astype(int)
    etyp = npz_eval['etyp'].flatten().astype(int)
    win  = np.arange(0, 4 * 250)
    X_eval = np.stack([s[pos + win, :].T for pos in epos[etyp == 783]])
    y_eval = loadmat(mat_path)['classlabel'].flatten() - 1

    X_train = resample_poly(X_train, up=64, down=125, axis=2)
    X_eval  = resample_poly(X_eval,  up=64, down=125, axis=2)

    # Per-channel standardization: fit on train, apply to both
    for ch in range(X_train.shape[1]):
        scaler = StandardScaler()
        X_train[:, ch, :] = scaler.fit_transform(X_train[:, ch, :])
        X_eval[:, ch, :]  = scaler.transform(X_eval[:, ch, :])

    X_train = torch.tensor(X_train[:, np.newaxis], dtype=torch.float32)
    X_eval  = torch.tensor(X_eval[:, np.newaxis],  dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_eval  = torch.tensor(y_eval,  dtype=torch.long)

    return X_train, y_train, X_eval, y_eval


In [ ]:
from scipy.io import loadmat
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

"""def train_subject(X_train, y_train, X_eval, y_eval, epochs=300, lr=0.001, batch_size=32):
    model = EEGNetModel(chans=22, classes=4, time_points=1000)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss_fn(model(X_batch), y_batch).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_eval).argmax(dim=1)
    return (preds == y_eval).float().mean().item()"""


'def train_subject(X_train, y_train, X_eval, y_eval, epochs=300, lr=0.001, batch_size=32):\n    model = EEGNetModel(chans=22, classes=4, time_points=1000)\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    loss_fn   = nn.CrossEntropyLoss()\n    loader    = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)\n\n    model.train()\n    for epoch in range(epochs):\n        for X_batch, y_batch in loader:\n            optimizer.zero_grad()\n            loss_fn(model(X_batch), y_batch).backward()\n            optimizer.step()\n\n    model.eval()\n    with torch.no_grad():\n        preds = model(X_eval).argmax(dim=1)\n    return (preds == y_eval).float().mean().item()'

In [ ]:
def train_subject(X_train, y_train, X_eval, y_eval, epochs=300, lr=0.001, batch_size=32):
    #model = EEGNetModel(chans=22, classes=4, time_points=1000).to(device)
    model = EEGNetModel(chans=22, classes=4, time_points=512, temp_kernel=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss_fn(model(X_batch), y_batch).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_eval.to(device)).argmax(dim=1).cpu()
    return (preds == y_eval).float().mean().item()


In [ ]:
#DATA_DIR = '/content/drive/MyDrive/BCI'
DATA_DIR = '/content'
accuracies = []

for idx in range(1, 10):
    X_train, y_train, X_eval, y_eval = subject_to_tensors(
        subjectData[f'subject{idx:02d}'],
        subjectDataEVAL[f'subject{idx:02d}'],
        f'{DATA_DIR}/A{idx:02d}E.mat'
    )
    acc = train_subject(X_train, y_train, X_eval, y_eval)
    accuracies.append(acc)
    print(f'Subject {idx:02d}: {acc:.4f}')

print(f'\nMean: {np.mean(accuracies):.4f}')

Subject 01: 0.7778
Subject 02: 0.5799
Subject 03: 0.8750
Subject 04: 0.6076
Subject 05: 0.6910
Subject 06: 0.5660
Subject 07: 0.7569
Subject 08: 0.8194
Subject 09: 0.7431

Mean: 0.7130


## EEGNet Baseline — Implementation Settings

**Implementation:** EEGNet-8,2 (Lawhern et al. 2018)

### Data
| Setting | Value |
|---|---|
| Dataset | BCI Competition IV Dataset 2a |
| Channels | 22 EEG (EOG dropped) |
| Sampling rate | 128 Hz (resampled from 250 Hz via resample_poly up=64, down=125) |
| Epoch window | 0–4 s from cue onset (events 769–772) = 512 samples |
| Train | A0xT.npz (288 trials/subject) |
| Test | A0xE.npz + true labels from A0xE.mat (288 trials/subject) |
| Artifacts | Not removed (all 288 trials kept) |

### Architecture
| Setting | Value |
|---|---|
| F1 (temporal filters) | 16 |
| D (depth multiplier) | 2 → 32 spatial filters |
| F2 (separable filters) | 32 |
| Temporal kernel | 64 samples = 500 ms at 128 Hz |
| Spatial kernel | (22, 1) depthwise over all channels |
| Pooling | AvgPool (1×8) after spatial, (1×16) after separable |
| Dropout | 0.5 after each pooling |
| Max-norm constraint | 1.0 on depthwise layer, 0.25 on FC layer |
| FC input size | 128 (= 512 // 128 × 32) |

### Training
| Setting | Value |
|---|---|
| Optimizer | Adam |
| Learning rate | 0.001 |
| Loss | CrossEntropyLoss |
| Epochs | 300 |
| Batch size | 32 |
| Hardware | GPU (Colab T4) |

### Results
| Subject | Accuracy |
|---|---|
| 01 | |
| 02 | |
| 03 | |
| 04 | |
| 05 | |
| 06 | |
| 07 | |
| 08 | |
| 09 | |
| **Mean** | |
